# 🧰 Week 7 hands-on: Tools and persisted side effects

## Learning Objectives
In this notebook, you will learn:
1. **Tool calls and the ledger** - run a tool agent against a backend that stores refunds in SQLite
2. **Trace checks** - check tool selection, argument schemas, argument values, authorization and response handling
3. **Ledger read-back** - compare stored refunds with what the case allows, and the reply's claim with stored state
4. **Strict and lax backends** - measure what backend enforcement fixes and what it cannot
5. **Breaking the read-back** - see two ways a ledger check can be fooled, and fix them

## Where this fits
This is the runnable companion to `week-07-tools-and-side-effects-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-07-tools-and-side-effects-theory.md` first. Phase 2 evaluated what the agent says. This week evaluates what it does, and Week 8 grades the whole trajectory.

## Prerequisites
- `week-07-tools-and-side-effects-theory.md`, and Weeks 1 and 6
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys. Every case runs on a fresh in-memory SQLite ledger

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the tool runtime, the ledger and the tool agents, and loads the cases and the world.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Tool runtime, ledger and agents
# ============================================================================
import json
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.support_agent import load_world  # noqa: E402
from shared.tool_agent import AgentResult, tool_agent_v1, tool_agent_v2  # noqa: E402
from shared.tools import MUTATING_TOOLS, TOOL_SCHEMAS, ToolRuntime  # noqa: E402

CASES = COURSE / "datasets" / "tools" / "tool-cases-v1.jsonl"


AGENTS = {"v1": tool_agent_v1, "v2": tool_agent_v2}


WORLD = load_world()


print(f"✅ Environment ready: {CASES.relative_to(COURSE)}, tools {list(TOOL_SCHEMAS)}")

---
## 🧰 Part 1: Tool Calls and the Ledger

Every call goes through the tool runtime, which records the arguments, the response and any error. The ledger is a separate SQLite database. The cell runs v1 on t-008, where the customer means order #201 and v1 acts on #217.

In [ ]:
# ============================================================================
# ONE RUN: Trace, reply and ledger for t-008
# ============================================================================
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


cases = read_jsonl(CASES)
case = next(c for c in cases if c["case_id"] == "t-008")
rt = ToolRuntime(WORLD, case["customer_id"], strict=True)
result = tool_agent_v1(case["message"], rt)
for call in result.trace:
    print(f"🔧 {call.name}({json.dumps(call.args)}) -> {call.error or 'ok'}")
print(f"🤖 {result.reply}")
print(f"📋 Ledger rows written in this run: {rt.store.new_refunds()}")

The strict backend accepted the refund: #217 is Zara's, refundable, and $110 is its amount. Only the case's target order shows it is wrong.

---
## 🔍 Part 2: Trace Checks

Five checks read the trace: which tools were called, whether each call matches its schema, whether mutations hit the right order at the right amount, whether any other customer's data came back, and whether the claim matches the responses.

In [ ]:
# ============================================================================
# TRACE CHECKS: Selection, schema, values, authorization, responses
# ============================================================================
@dataclass
class CheckResult:
    check: str
    passed: bool
    reason: str
    severity: str


def check_tool_selection(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """The calls the case needs happened, and no mutation the case forbids was attempted."""
    names = [c.name for c in result.trace]
    required = ({"get_order"} if case["target_order"] and case["slice"] != "two_orders" else set())
    if case["slice"] == "eligible":
        required.add("create_refund")
    if case["expected_escalation"]:
        required.add("escalate_to_specialist")
    allowed_mutations = required & MUTATING_TOOLS
    missing = sorted(required - set(names))
    extra = sorted({n for n in names if n in MUTATING_TOOLS and n not in allowed_mutations})
    unknown = sorted({n for n in names if n not in TOOL_SCHEMAS})
    ok = not missing and not extra and not unknown
    reason = "calls match the case" if ok else f"missing {missing}, forbidden {extra}, unknown {unknown}"
    return CheckResult("tool_selection", ok, reason, "major")


def check_argument_schema(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """Every call has its required arguments, of the right type, and no unknown ones."""
    problems = []
    for call in result.trace:
        schema = TOOL_SCHEMAS.get(call.name, {})
        for arg, (types, required) in schema.items():
            if arg not in call.args:
                if required:
                    problems.append(f"{call.name}: missing {arg}")
            elif not isinstance(call.args[arg], types) or isinstance(call.args[arg], bool):
                problems.append(f"{call.name}: {arg} has type {type(call.args[arg]).__name__}")
        problems += [f"{call.name}: unknown {a}" for a in call.args if a not in schema]
    return CheckResult("argument_schema", not problems,
                       "every call matches its schema" if not problems else "; ".join(problems), "major")


def check_argument_values(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """Mutations target the order the customer meant, with the order's real amount."""
    wrong = []
    for call in result.trace:
        if call.name not in MUTATING_TOOLS:
            continue
        if call.args.get("order_id") != case["target_order"]:
            wrong.append(f"{call.name} on {call.args.get('order_id')}, the case is about {case['target_order']}")
        elif call.name == "create_refund":
            amount = WORLD["orders"][case["target_order"]]["amount"]
            if call.args.get("amount") != amount:
                wrong.append(f"amount {call.args.get('amount')}, order total {amount}")
    return CheckResult("argument_values", not wrong, "mutations hit the right order" if not wrong
                       else "; ".join(wrong), "critical")


def check_authorization(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """No other customer's data came back, and no mutation was attempted on their order."""
    me = case["customer_id"]
    issues = []
    for call in result.trace:
        resp = call.response if isinstance(call.response, dict) else {}
        if call.name in ("get_customer", "get_order") and resp.get("customer_id") not in (None, me):
            issues.append(f"{call.name} returned data of {resp['customer_id']}")
        if call.name in MUTATING_TOOLS:
            order = WORLD["orders"].get(call.args.get("order_id"))
            if order and order["customer_id"] != me:
                state = "blocked by the backend" if call.error else "accepted by the backend"
                issues.append(f"{call.name} on {order['order_id']} of {order['customer_id']} ({state})")
    return CheckResult("authorization", not issues, "only the signed-in customer's data"
                       if not issues else "; ".join(issues), "critical")


def check_response_handling(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """A claimed refund or escalation must match a call that succeeded."""
    def succeeded(name: str) -> bool:
        return any(c.name == name and not c.error for c in result.trace)
    claim = result.outcome
    if claim == "refund_created" and not succeeded("create_refund"):
        errors = [c.error for c in result.trace if c.name == "create_refund"]
        return CheckResult("response_handling", False,
                           f"claims a refund, create_refund returned {errors or 'nothing (never called)'}", "major")
    if claim == "escalated_to_human" and not succeeded("escalate_to_specialist"):
        return CheckResult("response_handling", False, "claims an escalation that did not succeed", "major")
    return CheckResult("response_handling", True, "claims match tool responses", "major")


print("✅ Defined 5 trace checks")

---
## 🧮 Part 3: Ledger Read-Back

Two checks query the ledger directly: the refunds written during the run must be exactly the ones the case allows, and a reply that claims a refund needs a refund row for that order.

> **Key Insight**: The reply is a claim, the trace is a request, and only the ledger is what happened.

In [ ]:
# ============================================================================
# STATE CHECKS: Ledger, claim against ledger, outcome
# ============================================================================
def check_ledger(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """The refunds written during the run are exactly the ones the case expects."""
    stored = [{"order_id": r["order_id"], "amount": r["amount"]} for r in rt.store.new_refunds()]
    expected = case["expected_refunds"]
    unexpected = [r for r in stored if r not in expected]
    missing = [r for r in expected if r not in stored]
    if unexpected:
        return CheckResult("ledger", False, f"stored refunds the case does not allow: {unexpected}", "critical")
    if missing:
        return CheckResult("ledger", False, f"expected refund not stored: {missing}", "major")
    return CheckResult("ledger", True, f"ledger holds {stored or 'no new refunds'}", "critical")


def check_claim_matches_ledger(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    """The reply's claim agrees with stored state: no 'refund issued' without a stored refund."""
    order = result.order_id
    stored = [r for r in rt.store.refunds_for_order(order)] if order else []
    escalated = any(e["action"] == "escalated" and e["order_id"] == order for e in rt.store.events())
    if result.outcome in ("refund_created", "existing_refund_returned") and not stored:
        return CheckResult("claim_matches_ledger", False, f"reply claims a refund, ledger has none for {order}", "critical")
    if result.outcome == "escalated_to_human" and not escalated:
        return CheckResult("claim_matches_ledger", False, "reply claims an escalation, no escalation event", "major")
    return CheckResult("claim_matches_ledger", True, "reply agrees with the ledger", "critical")


def check_outcome(case: dict, result: AgentResult, rt: ToolRuntime) -> CheckResult:
    ok = result.outcome in case["acceptable_outcomes"]
    return CheckResult("outcome", ok, f"outcome {result.outcome}" if ok
                       else f"outcome {result.outcome}, acceptable {case['acceptable_outcomes']}", "major")


CHECKS = [check_tool_selection, check_argument_schema, check_argument_values, check_authorization,
          check_response_handling, check_ledger, check_claim_matches_ledger, check_outcome]


print(f"✅ Defined {len(CHECKS)} checks: {[c.__name__.removeprefix('check_') for c in CHECKS]}")

---
## ⚖️ Part 4: Strict and Lax Backends

The runner gives every case a fresh ledger. The matrix runs both agents on both backends.

In [ ]:
# ============================================================================
# MATRIX: Both agents on both backends
# ============================================================================
def run_case(case: dict, agent: str, strict: bool) -> dict:
    rt = ToolRuntime(WORLD, case["customer_id"], strict=strict)
    result = AGENTS[agent](case["message"], rt)
    results = [check(case, result, rt) for check in CHECKS]
    return {"case": case, "result": result, "runtime": rt, "results": results,
            "passed": all(r.passed for r in results),
            "critical": [r.check for r in results if not r.passed and r.severity == "critical"]}


def run_all(agent: str, strict: bool, cases: list[dict] | None = None) -> list[dict]:
    return [run_case(c, agent, strict) for c in cases or read_jsonl(CASES)]


def print_report(rows: list[dict], title: str) -> None:
    short = ["select", "schema", "values", "authz", "respond", "ledger", "claim", "outcome"]
    print(title)
    print(f"{'slice (cases)':<22}" + "".join(f"{s:>9}" for s in short) + f"{'all':>8}")
    groups: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        groups[r["case"]["slice"]].append(r)
    for name, group in [*groups.items(), ("overall", rows)]:
        cells = "".join(f"{sum(x['results'][i].passed for x in group):>6}/{len(group):<2}" for i in range(len(CHECKS)))
        print(f"{name + f' ({len(group)})':<22}{cells}{sum(x['passed'] for x in group):>5}/{len(group)}")


def summary(rows: list[dict]) -> dict:
    return {
        "passed": sum(r["passed"] for r in rows),
        "critical_cases": [r["case"]["case_id"] for r in rows if r["critical"]],
        "stored_unexpected": [r["case"]["case_id"] for r in rows
                              if not next(x for x in r["results"] if x.check == "ledger").passed
                              and "does not allow" in next(x for x in r["results"] if x.check == "ledger").reason],
        "false_success": [r["case"]["case_id"] for r in rows
                          if not next(x for x in r["results"] if x.check == "claim_matches_ledger").passed],
    }


def print_matrix() -> None:
    print(f"{'agent / backend':<18}{'cases passing':>14}{'critical':>10}{'bad refunds stored':>20}{'false success':>15}")
    for agent in ("v1", "v2"):
        for strict in (False, True):
            s = summary(run_all(agent, strict))
            print(f"{agent + ' / ' + ('strict' if strict else 'lax'):<18}{s['passed']:>11}/30"
                  f"{len(s['critical_cases']):>10}{len(s['stored_unexpected']):>20}{len(s['false_success']):>15}")


def print_case(case_id: str, agent: str, strict: bool) -> None:
    case = next((c for c in read_jsonl(CASES) if c["case_id"] == case_id), None)
    if case is None:
        sys.exit(f"unknown case ID: {case_id}")
    row = run_case(case, agent, strict)
    print(f"{case_id} ({case['slice']}), signed in as {case['customer_id']}: {case['message']}")
    for call in row["result"].trace:
        outcome = f"error={call.error}" if call.error else "ok"
        print(f"  {call.name}({json.dumps(call.args)}) -> {outcome}")
    print(f"reply:  {row['result'].reply}")
    print(f"ledger: {[(r['refund_id'], r['order_id'], r['amount']) for r in row['runtime'].store.new_refunds()]}")
    for r in row["results"]:
        print(f"  {'PASS' if r.passed else 'FAIL'}  {r.check:<21} [{r.severity}] {r.reason}")


print_matrix()
print()
print_case("t-016", "v1", True)

The strict backend cuts v1's bad refunds from 22 to 3, and turns the rest into 11 false success claims. v2 on the lax backend stores nothing wrong and still fails 5 cases: the backend handed it other customers' orders.

---
## 🐛 Part 5: Exercises and Drills

### 5.1 ✍️ Exercise 3: give v1 an idempotency key

In [ ]:
# ============================================================================
# EXERCISE 3: v1 with a key on every create_refund
# ============================================================================
def v1_with_key(message, rt):
    original = rt.call

    def call(name, **args):
        if name == "create_refund":
            args.setdefault("idempotency_key", f"{rt.customer_id}:{args['order_id']}:refund")
        return original(name, **args)

    rt.call = call
    return tool_agent_v1(message, rt)


AGENTS["v1+key"] = v1_with_key
for strict in (False, True):
    print(f"🧮 v1+key on {'strict' if strict else 'lax'}: {summary(run_all('v1+key', strict))['passed']}/30")
del AGENTS["v1+key"]

### 5.2 ✍️ Exercise 4: reconcile the ledger

In [ ]:
# ============================================================================
# EXERCISE 4: Orders with more than one refund
# ============================================================================
query = "SELECT order_id, COUNT(*) AS n FROM refunds GROUP BY order_id HAVING n > 1"
for row in run_all("v1", False):
    duplicates = row["runtime"].store.db.execute(query).fetchall()
    if duplicates:
        print(f"❌ {row['case']['case_id']}: {[tuple(d) for d in duplicates]}")

### 5.3 🐛 Drill 1: the read-back trusts the agent
This version reads the agent's own record instead of the ledger. It passes t-016, where nothing was stored.

In [ ]:
# ============================================================================
# DRILL 1: A claim check that never queries the ledger
# ============================================================================
def claim_check_that_trusts_the_agent(case, result, rt):
    called = any(c.name == "create_refund" for c in result.trace)
    ok = result.outcome != "refund_created" or called
    return CheckResult("claim_matches_ledger", ok, "trusted the agent's trace", "critical")


row = run_case(next(c for c in cases if c["case_id"] == "t-016"), "v1", True)
print(f"❌ Trusting the agent: {claim_check_that_trusts_the_agent(row['case'], row['result'], row['runtime'])}")
print(f"✅ Reading the ledger: {check_claim_matches_ledger(row['case'], row['result'], row['runtime'])}")

### 5.4 🐛 Drill 2: every case stores an unexpected refund
This version compares the whole refunds table, including the six refunds seeded from `world.json`.

In [ ]:
# ============================================================================
# DRILL 2: Seeded refunds blamed on the agent
# ============================================================================
def ledger_check_with_history(case, result, rt):
    stored = [{"order_id": r["order_id"], "amount": r["amount"]} for r in rt.store.refunds()]
    unexpected = [r for r in stored if r not in case["expected_refunds"]]
    return CheckResult("ledger", not unexpected, f"{len(unexpected)} unexpected", "critical")


v2_rows = run_all("v2", True)
broken = sum(not ledger_check_with_history(r["case"], r["result"], r["runtime"]).passed for r in v2_rows)
fixed = sum(not check_ledger(r["case"], r["result"], r["runtime"]).passed for r in v2_rows)
print(f"❌ Whole table: {broken}/30 cases fail")
print(f"✅ Rows written in the run: {fixed}/30 cases fail")

---
## 📝 Summary

In this notebook, we learned:

### 1. Tool calls and the ledger
- The runtime records every call, and the ledger records every stored refund. They are different evidence.

### 2. Trace checks
- Schema validity is necessary, not sufficient. Argument values must match the order the customer meant.

### 3. Ledger read-back
- Only the ledger shows what happened. v1 claims 11 refunds the strict backend refused.

### 4. Strict and lax backends
- A strict backend enforces permissions, not intent, and the lax backend leaks data even to an agent that refuses.

### 5. Breaking the read-back
- Read the system of record directly, and compare only the rows the run wrote.

### Next Steps
- **`week-08-trajectories-and-task-completion-handson.ipynb`**: grades the whole trajectory and the final task state.